# Graphiques RE2020 — bullet graphs

**Version 4.0 — 19/11/2026**

Application Jupyter/Voilà pour mettre en forme les résultats RE2020 par zone : titre, sous-titre, nom de zone, indicateurs, plusieurs groupes DH si nécessaire, puis informations de pied de graphique.

## Utilisation

1. En local : installez une fois les dépendances avec `pip install -r requirements.txt`, puis exécutez le notebook.
2. Avec Voilà/Binder : les dépendances sont installées au démarrage depuis `requirements.txt`. Le notebook ne lance plus `pip install`, ce qui supprime l'attente et le message de mise à jour de pip.
3. Téléchargez le modèle Excel depuis l'application, remplissez-le, puis importez le fichier `.xlsx` ou collez son contenu.
4. Pour plusieurs groupes DH, renseignez `DH_G1`, `DH_G2`, etc., avec les libellés `DH_G1_nom`, `DH_G2_nom`, etc.
5. L'ordre du graphique est : titre, sous-titre, zone, indicateurs, informations de bas de graphique.


## Historique des versions

| Version | Date | Évolutions |
|---|---|---|
| 3.0 | 10/11/2026 | Zones multiples, collage Excel, ICconstruction et exports. |
| 4.0 | 19/11/2026 | Déploiement Voilà/Binder, suppression du pip dans le notebook, vrai modèle XLSX téléchargeable, import XLSX, titre/sous-titre/pied, ordre graphique revu et DH multi-groupes. |


## 0. Dépendances
Les dépendances sont gérées par `requirements.txt`. Ne pas réinstaller les paquets à chaque lancement.


In [ ]:
# Aucun pip install ici : Binder/Voilà utilise requirements.txt.
# En local, exécuter une seule fois dans un terminal : pip install -r requirements.txt


## 1. Imports et constantes

In [ ]:
# ============================================================
# 1. IMPORTS ET CONSTANTES
# ============================================================
import io
import base64
import json
import math
import os
import re
import textwrap
import unicodedata
import functools
import html as _html
from dataclasses import dataclass, field, asdict, replace
from datetime import datetime

import matplotlib
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib import font_manager, patheffects
from matplotlib import colors as mcolors
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, FuncFormatter

import ipywidgets as W
from IPython.display import display
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

import copy

APP_NAME = "RE2020 Bullet Graphs"
APP_VERSION = "4.0"
APP_DATE = "2026-11-19"

INDICATOR_KEYS = ["Bbio", "CEP", "CEPnr", "DH", "ICénergie", "ICconstruction"]
# Nombre maximal de contributions : 8 par indicateur, 16 pour ICconstruction (13 lots + chantier)
MAX_CONTRIB = {k: 8 for k in INDICATOR_KEYS}
MAX_CONTRIB["ICconstruction"] = 16


def max_contrib(key):
    return MAX_CONTRIB.get(key, 8)

DEFAULT_UNITS = {
    "Bbio": "points",
    "CEP": "kWhep/(m².an)",
    "CEPnr": "kWhep/(m².an)",
    "DH": "°C.h",
    "ICénergie": "kg éq. CO2/m²",
    "ICconstruction": "kg éq. CO2/m²",
}

# Palette par défaut (sobre, contrastée, sans connotation conforme / non conforme)
PALETTE = ["#C8553D", "#3A7CA5", "#E9A23B", "#81B29A",
           "#5B5F97", "#8D6A9F", "#A8A9AD", "#2F4858",
           "#9C6644", "#6FB7C9", "#D98BA0", "#4A7C59",
           "#F2C57C", "#7D8597", "#B56576", "#355070"]

TEXT_COLOR = "#222222"
MUTED_COLOR = "#5F6368"
GRID_COLOR = "#DADCE0"
TARGET_COLOR = "#111111"
LEGEND_WRAP = 38          # nb de caractères avant retour à la ligne dans la légende
PREVIEW_DPI = 110         # résolution des aperçus (les exports utilisent le DPI choisi)


def _pick_font():
    """Choisit la première police disponible parmi une liste de polices de rapport."""
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in ["Arial", "Calibri", "Helvetica", "Liberation Sans", "DejaVu Sans"]:
        if name in available:
            return name
    return "DejaVu Sans"


FONT_NAME = _pick_font()


def _rc():
    """Contexte de style Matplotlib commun à l'aperçu et aux exports."""
    return matplotlib.rc_context({
        "font.family": "sans-serif",
        "font.sans-serif": [FONT_NAME, "DejaVu Sans"],
        "text.color": TEXT_COLOR,
        "axes.unicode_minus": False,
        "svg.fonttype": "path",      # rendu SVG identique partout (texte vectorisé)
        "pdf.fonttype": 42,          # polices TrueType incorporées dans le PDF
        "savefig.bbox": "standard",  # le recadrage est géré explicitement (voir export)
    })


print(f"{APP_NAME} v{APP_VERSION} ({APP_DATE}) | Matplotlib {matplotlib.__version__} | ipywidgets {W.__version__} | police : {FONT_NAME}")

## 2. Structure des données et jeu de démonstration (fictif)

In [ ]:
# ============================================================
# 2. STRUCTURE DES DONNÉES
# ============================================================
# - "état brut" (dict JSON) : ce que l'utilisateur saisit, y compris les
#   contributions inactives et les champs encore invalides ;
#   état = {"settings": {...}, "zones": [{"name": ..., "indicators": {clé: brut}}]}
# - objets validés (Indicator / Contribution / Settings) : ce qui est tracé.

@dataclass
class Contribution:
    label: str
    value: float
    color: str


@dataclass
class Indicator:
    key: str
    unit: str
    target: float
    contributions: list          # uniquement les contributions actives
    include: bool = True
    heading: str = ""            # titre du bloc (défaut : nom de l'indicateur)
    sub: str = ""                # texte gris à droite du titre (défaut : unité)

    @property
    def total(self):
        return float(sum(c.value for c in self.contributions))


@dataclass
class Settings:
    title: str = "Résultats RE2020"
    subtitle: str = ""
    footer: str = ""
    width_cm: float = 17.0
    height_overview_cm: float = 0.0   # 0 = automatique
    height_single_cm: float = 0.0     # 0 = automatique
    font_size: int = 9
    show_values: bool = True
    hide_zero_legend: bool = True     # masque les contributions nulles dans la légende
    common_scale: bool = True         # échelle commune dans les vues « par indicateur »
    decimals: int = 1
    background: str = "blanc"         # "blanc" ou "transparent"
    decimal_sep: str = ","
    dpi: int = 300
    export_dir: str = "exports_RE2020"
    file_prefix: str = "RE2020"
    formats: list = field(default_factory=lambda: ["png", "svg", "pdf"])

    @property
    def transparent(self):
        return self.background == "transparent"


# ---------- Correspondance avec le modèle Excel (clé | valeur)
# Clés normalisées : minuscules, sans suffixe « _widget ».
ICC_LOTS = [
    "Lot 1 – VRD",
    "Lot 2 – Fondations et infrastructure",
    "Lot 3 – Superstructure, maçonnerie",
    "Lot 4 – Couverture, étanchéité, charpente, zinguerie",
    "Lot 5 – Cloisonnement, doublage, plafonds suspendus, menuiseries intérieures",
    "Lot 6 – Façades et menuiseries extérieures",
    "Lot 7 – Revêtements des sols, murs et plafonds, chape, peintures, produits de décoration",
    "Lot 8 – CVC",
    "Lot 9 – Installations sanitaires",
    "Lot 10 – Réseaux d'énergie (courant fort)",
    "Lot 11 – Réseaux de communication (courant faible)",
    "Lot 12 – Appareils élévateurs et autres équipements de transport intérieur",
    "Lot 13 – Équipement de production locale d'électricité",
]

EXCEL_MAP = {
    "Bbio": {"show": "show_bbio", "max": "bbiomax", "contribs": [
        ("bbiochaud", "Besoins de chauffage"),
        ("bbiofroid", "Besoins de refroidissement"),
        ("bbioecl", "Besoins d'éclairage")]},
    "CEP": {"show": "show_cep", "max": "cepmax", "contribs": [
        ("cepch", "Chauffage"), ("cepfr", "Refroidissement"), ("cepecs", "ECS"),
        ("cepecl", "Éclairage"), ("cepauxv", "Auxiliaires de ventilation"),
        ("cepauxd", "Auxiliaires de distribution"), ("cepdep", "Déplacements")]},
    "CEPnr": {"show": "show_cepnr", "max": "cepnrmax", "contribs": [
        ("cepnrch", "Chauffage"), ("cepnrfr", "Refroidissement"), ("cepnrecs", "ECS"),
        ("cepnrecl", "Éclairage"), ("cepnrauxv", "Auxiliaires de ventilation"),
        ("cepnrauxd", "Auxiliaires de distribution"), ("cepnrdep", "Déplacements")]},
    "DH": {"show": "show_dh", "max": "dhmax", "contribs": [
        (f"dh_g{i}", f"Groupe {i}") for i in range(1, 9)]},
    "ICénergie": {"show": "show_ice", "max": "icemax", "contribs": [
        ("iceelec", "Électricité"), ("icebois", "Bois"), ("icercu", "Réseau de chaleur urbain"),
        ("icefioul", "Fioul"), ("icegaz", "Gaz")]},
    "ICconstruction": {"show": "show_icc", "max": "iccmax", "contribs":
        [(f"icc_lot{i + 1:02d}", lab) for i, lab in enumerate(ICC_LOTS)]
        + [("icc_chantier", "Chantier")]},
}

# Orthographe affichée des clés (identique au modèle Excel)
EXCEL_KEY_NAMES = {k.lower(): k + "_widget" for k in [
    "titre", "soustitre", "infos", "zone", "cepnr_equals_cep",
    "BbioChaud", "BbioFroid", "BbioEcl", "Bbiomax", "show_bbio",
    "CEPch", "CEPfr", "CEPecs", "CEPecl", "CEPauxV", "CEPauxD", "CEPdep", "CEPmax", "show_cep",
    "CEPnrch", "CEPnrfr", "CEPnrecs", "CEPnrecl", "CEPnrauxV", "CEPnrauxD", "CEPnrdep", "CEPnrmax",
    "show_cepnr", *[f"DH_G{i}" for i in range(1, 9)], *[f"DH_G{i}_nom" for i in range(1, 9)], "DHmax", "show_dh",
    "IcEelec", "IcEbois", "IcErcu", "IcEfioul", "IcEgaz", "IcEmax", "show_ice",
    *[f"IcC_lot{i:02d}" for i in range(1, 14)], "IcC_chantier", "IcCmax", "show_icc"]}


def excel_name(k):
    return EXCEL_KEY_NAMES.get(k, f"{k}_widget")


def _spec_color(key, ck):
    cks = [c[0] for c in EXCEL_MAP[key]["contribs"]]
    return PALETTE[cks.index(ck) % len(PALETTE)] if ck in cks else PALETTE[0]


def blank_contrib(i):
    return {"key": "", "label": "", "value": "0", "color": PALETTE[i % len(PALETTE)]}


def _pad(contribs, key):
    contribs = list(contribs)[:max_contrib(key)]
    return contribs + [blank_contrib(i) for i in range(len(contribs), max_contrib(key))]


def template_raw_indicator(key):
    """Indicateur vierge pré-rempli avec les contributions du modèle Excel (valeurs à 0)."""
    contribs = [{"key": ck, "label": lab, "value": "0", "color": _spec_color(key, ck)}
                for ck, lab in EXCEL_MAP[key]["contribs"]]
    return {"unit": DEFAULT_UNITS[key], "target": "", "n_active": len(contribs),
            "include": True, "contributions": _pad(contribs, key)}


def new_zone(name):
    return {"name": name, "indicators": {k: template_raw_indicator(k) for k in INDICATOR_KEYS}}


def default_state():
    return {"version": 2, "settings": asdict(Settings()), "zones": [new_zone("Zone 1")],
            "current_zone": 0}


def _num_to_str(v):
    if isinstance(v, bool) or v is None:
        return ""
    if isinstance(v, (int, float)):
        return f"{v:g}" if abs(v) < 1e15 else str(v)
    return str(v)


def raw_from_values(key, target, values, include=True):
    """Indicateur brut à partir d'un dict {clé Excel: valeur} (ordre du modèle)."""
    labels = dict(EXCEL_MAP[key]["contribs"])
    contribs = [{"key": ck, "label": labels[ck], "value": _num_to_str(values[ck]),
                 "color": _spec_color(key, ck)}
                for ck, _ in EXCEL_MAP[key]["contribs"] if ck in values]
    return {"unit": DEFAULT_UNITS[key], "target": _num_to_str(target),
            "n_active": max(1, len(contribs)), "include": include,
            "contributions": _pad(contribs or [blank_contrib(0)], key)}


def demo_state():
    """Jeu de données de DÉMONSTRATION — deux zones, valeurs entièrement FICTIVES."""
    st = default_state()
    st["settings"]["title"] = "Projet fictif — données de démonstration"
    za = {
        "Bbio": raw_from_values("Bbio", 63, {"bbiochaud": 34.2, "bbiofroid": 17.5, "bbioecl": 11.0}),
        "CEP": raw_from_values("CEP", 75, {"cepch": 21.4, "cepfr": 9.8, "cepecs": 3.1, "cepecl": 12.6,
                                           "cepauxv": 8.7, "cepauxd": 2.4}),
        "CEPnr": raw_from_values("CEPnr", 65, {"cepnrch": 26.3, "cepnrfr": 11.2, "cepnrecs": 3.9,
                                               "cepnrecl": 15.5, "cepnrauxv": 10.8, "cepnrauxd": 2.0}),
        "DH": raw_from_values("DH", 1250, {"dh_g1": 742, "dh_g2": 618}),
        "ICénergie": raw_from_values("ICénergie", 160, {"iceelec": 98.0, "icercu": 52.4}),
        "ICconstruction": raw_from_values("ICconstruction", 740, {
            "icc_lot01": 25, "icc_lot02": 95, "icc_lot03": 210, "icc_lot04": 48, "icc_lot05": 62,
            "icc_lot06": 120, "icc_lot07": 55, "icc_lot08": 88, "icc_lot09": 14, "icc_lot10": 30,
            "icc_lot11": 9, "icc_lot12": 12, "icc_chantier": 18}),
    }
    zb = {
        "Bbio": raw_from_values("Bbio", 80, {"bbiochaud": 41.0, "bbiofroid": 22.3, "bbioecl": 9.5}),
        "CEP": raw_from_values("CEP", 110, {"cepch": 30.2, "cepfr": 14.1, "cepecs": 18.4, "cepecl": 10.3,
                                            "cepauxv": 12.9, "cepauxd": 3.0}),
        "CEPnr": raw_from_values("CEPnr", 95, {"cepnrch": 27.5, "cepnrfr": 14.1, "cepnrecs": 16.0,
                                               "cepnrecl": 10.3, "cepnrauxv": 12.9, "cepnrauxd": 3.0}),
        "DH": raw_from_values("DH", 1250, {"dh_g1": 1105}),
        "ICénergie": raw_from_values("ICénergie", 210, {"iceelec": 121.0, "icegaz": 64.5}),
        "ICconstruction": raw_from_values("ICconstruction", 800, {
            "icc_lot01": 31, "icc_lot02": 88, "icc_lot03": 185, "icc_lot04": 40, "icc_lot05": 58,
            "icc_lot06": 97, "icc_lot07": 61, "icc_lot08": 104, "icc_lot09": 26, "icc_lot10": 28,
            "icc_lot11": 8, "icc_lot12": 0, "icc_chantier": 21}),
    }
    za["DH"]["contributions"][0]["label"] = "Groupe bureaux"
    za["DH"]["contributions"][1]["label"] = "Groupe salles de réunion"
    zb["DH"]["contributions"][0]["label"] = "Groupe restauration"
    st["zones"] = [{"name": "Zone A – Bureaux (fictive)", "indicators": za},
                   {"name": "Zone B – Restauration (fictive)", "indicators": zb}]
    return st


def _normalize_raw_indicator(key, raw_in):
    raw = template_raw_indicator(key)
    if not isinstance(raw_in, dict):
        return raw
    raw["unit"] = str(raw_in.get("unit", raw["unit"]) or "")
    raw["target"] = _num_to_str(raw_in.get("target", ""))
    try:
        n = int(raw_in.get("n_active", 1))
    except (TypeError, ValueError):
        n = 1
    raw["n_active"] = min(max(n, 1), max_contrib(key))
    raw["include"] = bool(raw_in.get("include", True))
    contribs = raw_in.get("contributions", [])
    if isinstance(contribs, list):
        out = []
        for i, c in enumerate(contribs[:max_contrib(key)]):
            if not isinstance(c, dict):
                c = {}
            out.append({"key": str(c.get("key", "") or "").strip().lower(),
                        "label": str(c.get("label", "") or ""),
                        "value": _num_to_str(c.get("value", "")),
                        "color": str(c.get("color", PALETTE[i % len(PALETTE)]) or PALETTE[i % len(PALETTE)])})
        raw["contributions"] = _pad(out, key)
    return raw


def normalize_state(data):
    """Fusionne un état chargé (JSON) avec l'état par défaut : tolère les clés manquantes
    et relit les projets de l'ancienne version (une seule zone)."""
    if not isinstance(data, dict):
        raise ValueError("le fichier ne contient pas un objet JSON de projet.")
    state = default_state()
    s_in = data.get("settings", {})
    if isinstance(s_in, dict):
        for k in state["settings"]:
            if k in s_in:
                state["settings"][k] = s_in[k]
    zones_in = data.get("zones")
    if not isinstance(zones_in, list) or not zones_in:
        zones_in = [{"name": "Zone 1", "indicators": data.get("indicators", {})}]
    zones = []
    for i, z in enumerate(zones_in):
        if not isinstance(z, dict):
            continue
        inds = z.get("indicators", {}) if isinstance(z.get("indicators"), dict) else {}
        name = str(z.get("name", "") or "").strip() or f"Zone {i + 1}"
        zones.append({"name": name,
                      "indicators": {k: (_normalize_raw_indicator(k, inds[k]) if k in inds
                                         else template_raw_indicator(k)) for k in INDICATOR_KEYS}})
    state["zones"] = zones or [new_zone("Zone 1")]
    try:
        cur = int(data.get("current_zone", 0))
    except (TypeError, ValueError):
        cur = 0
    state["current_zone"] = min(max(cur, 0), len(state["zones"]) - 1)
    return state

## 3. Validation des saisies

In [ ]:
# ============================================================
# 3. VALIDATION
# ============================================================

@dataclass
class ValidationResult:
    indicator: object = None      # Indicator si valide, sinon None
    errors: list = field(default_factory=list)
    warnings: list = field(default_factory=list)

    @property
    def ok(self):
        return not self.errors and self.indicator is not None


def parse_number(text):
    """Convertit une saisie en nombre. Accepte la virgule décimale et les espaces.
    Retourne (valeur, message_erreur)."""
    if isinstance(text, bool):
        return None, "valeur booléenne"
    if isinstance(text, (int, float)):
        return (float(text), None) if math.isfinite(text) else (None, "valeur non finie")
    s = "" if text is None else str(text)
    s = s.strip().replace("\u202f", "").replace("\u00a0", "").replace(" ", "")
    if not s:
        return None, "champ vide"
    s = s.replace(",", ".")
    try:
        v = float(s)
    except ValueError:
        return None, f"« {str(text).strip()} » n'est pas un nombre"
    if not math.isfinite(v):
        return None, "valeur non finie"
    return v, None


def normalize_color(c, fallback):
    try:
        if c and mcolors.is_color_like(c):
            return mcolors.to_hex(c), True
    except (ValueError, TypeError):
        pass
    return fallback, False


def validate_indicator(key, raw):
    res = ValidationResult()
    try:
        n = int(raw.get("n_active", 1))
    except (TypeError, ValueError):
        n = 1
    n = min(max(n, 1), max_contrib(key))

    unit = str(raw.get("unit", "") or "").strip()
    if not unit:
        res.warnings.append("unité non renseignée")

    target, err = parse_number(raw.get("target", ""))
    if err:
        res.errors.append(f"Valeur maximale : {err}.")
    elif target <= 0:
        res.errors.append("Valeur maximale : elle doit être strictement positive.")

    contribs = []
    for i in range(n):
        c = raw["contributions"][i]
        label = str(c.get("label", "") or "").strip() or f"Contribution {i + 1}"
        v, err = parse_number(c.get("value", ""))
        if err:
            res.errors.append(f"Contribution {i + 1} ({label}) : {err}.")
            continue
        if v < 0:
            res.errors.append(f"Contribution {i + 1} ({label}) : valeur négative non admise.")
            continue
        color, ok = normalize_color(c.get("color"), PALETTE[i % len(PALETTE)])
        if not ok:
            res.warnings.append(f"Contribution {i + 1} : couleur invalide, couleur par défaut utilisée.")
        contribs.append(Contribution(label, v, color))

    if not res.errors:
        res.indicator = Indicator(key, unit, target, contribs, bool(raw.get("include", True)))
    return res


def validate_settings(raw):
    """Contrôle et borne les paramètres généraux (jamais bloquant)."""
    s = Settings()
    warnings = []

    def _num(name, cast, lo, hi):
        val = raw.get(name, getattr(s, name))
        try:
            val = cast(val)
            if not math.isfinite(float(val)):
                raise ValueError
        except (TypeError, ValueError):
            warnings.append(f"Paramètre « {name} » invalide : valeur par défaut utilisée.")
            return getattr(s, name)
        if val < lo or val > hi:
            warnings.append(f"Paramètre « {name} » ramené dans l'intervalle [{lo} ; {hi}].")
            val = min(max(val, lo), hi)
        return val

    s.title = str(raw.get("title", s.title) or "")
    s.subtitle = str(raw.get("subtitle", s.subtitle) or "")
    s.footer = str(raw.get("footer", s.footer) or "")
    s.width_cm = _num("width_cm", float, 8.0, 60.0)
    s.height_overview_cm = _num("height_overview_cm", float, 0.0, 100.0)
    s.height_single_cm = _num("height_single_cm", float, 0.0, 100.0)
    s.font_size = _num("font_size", int, 6, 20)
    s.decimals = _num("decimals", int, 0, 4)
    s.dpi = _num("dpi", int, 300, 1200)
    s.show_values = bool(raw.get("show_values", True))
    s.hide_zero_legend = bool(raw.get("hide_zero_legend", True))
    s.common_scale = bool(raw.get("common_scale", True))
    s.background = "transparent" if raw.get("background") == "transparent" else "blanc"
    s.decimal_sep = "." if raw.get("decimal_sep") == "." else ","
    s.export_dir = str(raw.get("export_dir", s.export_dir) or "").strip() or "exports_RE2020"
    s.file_prefix = str(raw.get("file_prefix", s.file_prefix) or "").strip() or "RE2020"
    fmts = raw.get("formats", s.formats)
    if not isinstance(fmts, (list, tuple)):
        fmts = s.formats
    s.formats = [f for f in ("png", "svg", "pdf") if f in fmts]
    return s, warnings

## 4. Fonctions de tracé

In [ ]:
# ============================================================
# 4. FONCTIONS DE TRACÉ (Matplotlib, API objet — aucune figure pyplot globale)
# ============================================================

BAR_HALF = 0.28          # demi-hauteur de la barre (axe y de -0.5 à 0.5)


def fmt_num(v, decimals, sep):
    s = f"{v:,.{int(decimals)}f}"
    if sep == ",":
        s = s.replace(",", "\u00a0").replace(".", ",")
    return s


def fmt_target(v, decimals, sep):
    d = 0 if abs(v - round(v)) < 1e-9 else decimals
    return fmt_num(v, d, sep)


def _font_sizes(base):
    return {"title": base * 1.45, "head": base * 1.12, "ann": base * 0.92,
            "tick": base * 0.85, "leg": base * 0.88, "seg": base * 0.82}


def _text_color_for(bg):
    """Noir ou blanc selon le meilleur contraste (WCAG) avec la couleur du segment."""
    r, g, b = mcolors.to_rgb(bg)
    lin = [c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4 for c in (r, g, b)]
    lum = 0.2126 * lin[0] + 0.7152 * lin[1] + 0.0722 * lin[2]
    return "#1A1A1A" if (lum + 0.05) / 0.05 >= 1.05 / (lum + 0.05) else "#FFFFFF"


def _new_figure(w_in, h_in):
    fig = Figure(figsize=(w_in, h_in), dpi=100)
    FigureCanvasAgg(fig)
    return fig


def _legend_kwargs(fs):
    return dict(frameon=False, fontsize=fs, handlelength=1.1, handleheight=0.9,
                handletextpad=0.5, columnspacing=1.8, labelspacing=0.45,
                borderpad=0.1, borderaxespad=0.0)


def _legend_entries(ind, ncol):
    """Ordonne les entrées pour une lecture ligne par ligne (même ordre que la barre)."""
    handles = [Patch(facecolor=c.color, edgecolor="none") for c in ind.contributions]
    labels = [textwrap.fill(c.label, LEGEND_WRAP) for c in ind.contributions]
    n = len(labels)
    ncol = max(1, min(ncol, n))
    nrows, n_large = divmod(n, ncol)
    rows_per_col = [nrows + 1] * n_large + [nrows] * (ncol - n_large)
    order = [r * ncol + c for c in range(ncol) for r in range(rows_per_col[c])]
    return [handles[i] for i in order], [labels[i] for i in order]


def _measure_legend(ind, avail_w_in, fs):
    """Nombre de colonnes maximal tenant dans la largeur, puis hauteur réelle mesurée."""
    fig = _new_figure(max(avail_w_in * 2, 6), 6)
    r = fig.canvas.get_renderer()
    n = len(ind.contributions)
    if n == 0:
        return 1, 0.0

    def size(ncol):
        h, l = _legend_entries(ind, ncol)
        leg = fig.legend(h, l, ncol=ncol, loc="upper left", **_legend_kwargs(fs))
        bb = leg.get_window_extent(r)
        leg.remove()
        return bb.width / fig.dpi, bb.height / fig.dpi

    best = 1
    for ncol in range(n, 0, -1):
        if size(ncol)[0] <= avail_w_in:
            best = ncol
            break
    rows = math.ceil(n / best)
    best = math.ceil(n / rows)            # colonnes équilibrées pour ce nombre de lignes
    w, h = size(best)
    return best, h


def _block_metrics(ind, s, avail_w):
    fs = _font_sizes(s.font_size)
    ncol, leg_h = _measure_legend(ind, avail_w, fs["leg"])
    m = {
        "ncol": ncol,
        "head_h": fs["head"] * 1.5 / 72,
        "ann_h": fs["ann"] * 1.6 / 72,
        "bar_h": max(0.30, s.font_size * 2.6 / 72),
        "tick_h": fs["tick"] * 2.2 / 72,
        "leg_gap": 0.06,
        "leg_h": leg_h,
    }
    m["total"] = sum(m[k] for k in ("head_h", "ann_h", "bar_h", "tick_h", "leg_gap", "leg_h"))
    return m


def _tick_formatter(xmax, sep):
    loc = MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10], min_n_ticks=3)
    ticks = [t for t in loc.tick_values(0, xmax) if 0 <= t <= xmax * 1.0001]
    d = 0
    for d in range(0, 5):
        if all(abs(t * 10 ** d - round(t * 10 ** d)) < 1e-6 for t in ticks):
            break
    return loc, FuncFormatter(lambda v, pos: fmt_num(v, d, sep))


def _draw_bullet(fig, ax, ind, s, m, min_xmax=0.0):
    """Trace un bullet graph : barre empilée + repère de cible + annotations."""
    fs = _font_sizes(s.font_size)
    r = fig.canvas.get_renderer()
    px_per_pt = fig.dpi / 72
    total, target = ind.total, ind.target

    ax.set_ylim(-0.5, 0.5)
    ax.set_yticks([])
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color("#9AA0A6")
    ax.spines["bottom"].set_linewidth(0.8)
    ax.set_facecolor("none")
    ax.set_axisbelow(True)
    ax.grid(axis="x", color=GRID_COLOR, linewidth=0.6)

    # Titre du bloc (gras) + complément (gris : unité, zone…)
    head = ax.annotate(ind.heading or ind.key, xy=(0, 1), xycoords="axes fraction",
                       xytext=(0, m["ann_h"] * 72), textcoords="offset points",
                       ha="left", va="bottom", fontsize=fs["head"], fontweight="bold",
                       color=TEXT_COLOR)
    sub = ind.sub if ind.sub else (f"({ind.unit})" if ind.unit else "")
    if sub:
        w_head_pt = head.get_window_extent(r).width / px_per_pt
        ax.annotate(sub, xy=(0, 1), xycoords="axes fraction",
                    xytext=(w_head_pt + 5, m["ann_h"] * 72), textcoords="offset points",
                    ha="left", va="bottom", fontsize=fs["head"] * 0.9, color=MUTED_COLOR)

    # Segments empilés
    segments, left = [], 0.0
    for c in ind.contributions:
        if c.value > 0:
            ax.barh(0, c.value, left=left, height=2 * BAR_HALF, color=c.color,
                    edgecolor="white", linewidth=0.8, zorder=2)
        segments.append((left, c.value, c))
        left += c.value

    # Repère de cible : trait foncé avec liseré blanc (visible sur tout segment)
    ax.plot([target, target], [-0.42, 0.5], color=TARGET_COLOR, linewidth=1.8,
            solid_capstyle="butt", zorder=5, clip_on=False,
            path_effects=[patheffects.withStroke(linewidth=3.8, foreground="white")])

    total_txt = ax.annotate(f"Total : {fmt_num(total, s.decimals, s.decimal_sep)}",
                            xy=(total, 0), xytext=(6, 0), textcoords="offset points",
                            ha="left", va="center", fontsize=fs["ann"], fontweight="bold",
                            color=TEXT_COLOR, zorder=6, annotation_clip=False)
    tgt_txt = ax.annotate(f"Max : {fmt_target(target, s.decimals, s.decimal_sep)}",
                          xy=(target, 1), xycoords=("data", "axes fraction"),
                          xytext=(0, 2), textcoords="offset points",
                          ha="center", va="bottom", fontsize=fs["ann"],
                          color=TARGET_COLOR, zorder=6, annotation_clip=False)

    # Échelle propre à l'indicateur : 0 -> max(total, cible) + place pour les textes
    M = max(total, target)
    ax.set_xlim(0, M * 1.04)
    w_px = ax.bbox.width
    w_tot = total_txt.get_window_extent(r).width
    w_tgt = tgt_txt.get_window_extent(r).width
    off, pad = 6 * px_per_pt, 4 * px_per_pt

    def _needs_shift(xmax):
        # Si la cible tombe juste après la barre, le total est décalé à droite de la cible
        sc = w_px / xmax
        return total <= target and (target - total) * sc < off + w_tot + off

    xmax = M * 1.04
    for _ in range(20):
        anchor = target if _needs_shift(xmax) else total
        d1 = max(w_px - off - w_tot - pad, 0.2 * w_px)
        d2 = max(w_px - w_tgt / 2 - pad, 0.2 * w_px)
        new = max(xmax, anchor * w_px / d1, target * w_px / d2)
        if new <= xmax * (1 + 1e-9):
            break
        xmax = new
    xmax = max(xmax, min_xmax)          # échelle commune éventuelle
    anchor = target if _needs_shift(xmax) else total
    total_txt.xy = (anchor, 0)
    ax.set_xlim(0, xmax)
    sc = w_px / xmax

    # Annotation de cible : alignée à gauche si elle déborderait à gauche
    if target * sc < w_tgt / 2 + pad:
        tgt_txt.set_ha("left")
        tgt_txt.set_position((-1, 2))

    # Valeurs dans les segments : seulement si lisibles et non traversées par la cible
    if s.show_values:
        bar_px = ax.bbox.height * 2 * BAR_HALF
        for lft, v, c in segments:
            if v <= 0:
                continue
            t = ax.text(lft + v / 2, 0, fmt_num(v, s.decimals, s.decimal_sep),
                        ha="center", va="center", fontsize=fs["seg"],
                        color=_text_color_for(c.color), zorder=4)
            bb = t.get_window_extent(r)
            fits = bb.width + 2 * pad <= v * sc and bb.height <= bar_px * 0.9
            hits_target = abs(target * sc - (lft + v / 2) * sc) < bb.width / 2 + 3 * px_per_pt
            if not fits or hits_target:
                t.remove()

    loc, fmt = _tick_formatter(xmax, s.decimal_sep)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)
    ax.tick_params(axis="x", labelsize=fs["tick"], length=3, width=0.6,
                   color="#9AA0A6", labelcolor=MUTED_COLOR, pad=2)
    return xmax


def _wrap_title(title, avail_w_in, fontsize):
    """Renvoie le titre découpé en lignes pour tenir dans la largeur disponible."""
    title = title.strip()
    if not title:
        return ""
    fig = _new_figure(max(avail_w_in * 3, 6), 2)
    r = fig.canvas.get_renderer()
    for width in range(len(title), 9, -2):
        wrapped = textwrap.fill(title, width)
        t = fig.text(0, 0.5, wrapped, fontsize=fontsize, fontweight="bold")
        ok = t.get_window_extent(r).width / fig.dpi <= avail_w_in
        t.remove()
        if ok:
            return wrapped
    return textwrap.fill(title, 10)


def _build(indicators, s, height_cm, title, subtitle, zone_name, footer, min_xmax):
    with _rc():
        fs = _font_sizes(s.font_size)
        Wd = s.width_cm / 2.54
        margin_l, margin_r = 0.30, 0.40
        avail = Wd - margin_l - margin_r
        metrics = [_block_metrics(ind, s, avail) for ind in indicators]
        n = len(indicators)
        top_pad, bottom_pad = 0.12, 0.10
        title = _wrap_title(title, avail, fs["title"])
        subtitle = _wrap_title(subtitle, avail, fs["head"])
        zone_name = _wrap_title(zone_name, avail, fs["head"])
        footer = _wrap_title(footer, avail, fs["ann"])
        title_h = (fs["title"] * (1.2 * (title.count("\n") + 1) + 0.35) / 72) if title else 0.0
        subtitle_h = (fs["head"] * (1.2 * (subtitle.count("\n") + 1) + 0.25) / 72) if subtitle else 0.0
        zone_h = (fs["head"] * 1.7 / 72) if zone_name else 0.0
        footer_h = (fs["ann"] * (1.25 * (footer.count("\n") + 1) + 0.4) / 72) if footer else 0.0
        gap = max(0.20, s.font_size * 2.2 / 72)
        h_req = top_pad + title_h + subtitle_h + zone_h + sum(m["total"] for m in metrics) + gap * (n - 1) + footer_h + bottom_pad

        H, note = h_req, None
        if height_cm and height_cm > 0:
            h_user = height_cm / 2.54
            if h_user >= h_req:
                H = h_user
                extra = h_user - h_req
                if n > 1:
                    gap += extra / (n - 1)
                else:
                    top_pad += extra / 2
            else:
                note = (f"hauteur demandée ({height_cm:.1f} cm) insuffisante : "
                        f"portée à {h_req * 2.54:.1f} cm pour éviter les chevauchements")

        fig = _new_figure(Wd, H)
        y = H - top_pad
        if title_h:
            fig.text(margin_l / Wd, y / H, title, ha="left", va="top", linespacing=1.15,
                     fontsize=fs["title"], fontweight="bold", color=TEXT_COLOR)
            y -= title_h
        if subtitle_h:
            fig.text(margin_l / Wd, y / H, subtitle, ha="left", va="top", linespacing=1.15,
                     fontsize=fs["head"], color=MUTED_COLOR)
            y -= subtitle_h
        if zone_h:
            fig.text(margin_l / Wd, y / H, zone_name, ha="left", va="top",
                     fontsize=fs["head"], fontweight="bold", color=TEXT_COLOR)
            y -= zone_h
        xmaxes = []
        for ind, m in zip(indicators, metrics):
            ax_top = y - m["head_h"] - m["ann_h"]
            ax_bottom = ax_top - m["bar_h"]
            ax = fig.add_axes([margin_l / Wd, ax_bottom / H, avail / Wd, m["bar_h"] / H])
            xmaxes.append(_draw_bullet(fig, ax, ind, s, m, min_xmax))
            leg_top = ax_bottom - m["tick_h"] - m["leg_gap"]
            if ind.contributions:
                h, l = _legend_entries(ind, m["ncol"])
                fig.legend(h, l, ncol=m["ncol"], loc="upper left",
                           bbox_to_anchor=(margin_l / Wd, leg_top / H),
                           bbox_transform=fig.transFigure, **_legend_kwargs(fs["leg"]))
            y = leg_top - m["leg_h"] - gap
        if footer_h:
            fig.text(margin_l / Wd, bottom_pad / H, footer, ha="left", va="bottom",
                     fontsize=fs["ann"], color=MUTED_COLOR, linespacing=1.2)
        return fig, note, xmaxes


def _for_plot(ind, s):
    """Masque les contributions nulles de la légende si l'option est active (total inchangé)."""
    if s.hide_zero_legend:
        return replace(ind, contributions=[c for c in ind.contributions if c.value > 0])
    return ind


def build_figure(indicators, s, height_cm=0.0, title="", subtitle="", zone_name="", footer="", common_scale=False):
    """Construit une figure avec un bloc par indicateur. Retourne (figure, note sur la hauteur).
    common_scale : même axe pour tous les blocs (comparaison d'un indicateur entre zones)."""
    indicators = [_for_plot(i, s) for i in indicators]
    fig, note, xmaxes = _build(indicators, s, height_cm, title, subtitle, zone_name, footer, 0.0)
    if common_scale and len(xmaxes) > 1 and max(xmaxes) > min(xmaxes) * (1 + 1e-9):
        fig, note, _ = _build(indicators, s, height_cm, title, subtitle, zone_name, footer, max(xmaxes))
    return fig, note


def _join_title(*parts):
    return " — ".join(p.strip() for p in parts if p and p.strip())


def plan_views(zones_valid, s):
    """Liste des graphiques à produire à partir des zones validées.
    zones_valid : [(nom de zone, {indicateur: ValidationResult (indicateurs inclus)})]
    Chaque vue : dict(group, caption, basename, title, items, height, common)."""
    nz = len(zones_valid)
    prefix = safe_filename(s.file_prefix)
    views = []
    tags = [f"Z{i + 1}_{safe_filename(n)}" for i, (n, _) in enumerate(zones_valid)]

    # 1. Vue d'ensemble : une figure par zone, tous les indicateurs inclus
    for (zname, results), tag in zip(zones_valid, tags):
        if not results or any(not r.ok for r in results.values()):
            continue
        items = []
        for k, r in results.items():
            if k == "DH":
                for c in r.indicator.contributions:
                    if c.value > 0:
                        items.append(Indicator("DH", r.indicator.unit, r.indicator.target,
                                               [Contribution(c.label, c.value, c.color)], True,
                                               heading=f"DH — {c.label}", sub=""))
            else:
                items.append(replace(r.indicator, heading=k, sub=""))
        views.append(dict(group="zone", caption=f"Vue d'ensemble — {zname}",
                          basename=f"{prefix}_ensemble" + (f"_{tag}" if nz > 1 else ""),
                          title=s.title, subtitle=s.subtitle, zone_name=zname, footer=s.footer,
                          items=items, height=s.height_overview_cm, common=False))

    # 2. Vue par indicateur : un même indicateur, une barre par zone (à partir de 2 zones)
    if nz > 1:
        for k in INDICATOR_KEYS:
            items = [replace(res[k].indicator, heading=zname,
                             sub=f"({res[k].indicator.unit})" if res[k].indicator.unit else "")
                     for zname, res in zones_valid if k in res and res[k].ok]
            if len(items) >= 2:
                views.append(dict(group="indicator", caption=f"{k} — comparaison des zones",
                                  basename=f"{prefix}_{safe_filename(k)}_zones",
                                  title=s.title, subtitle=s.subtitle, zone_name=f"Comparaison des zones — {k}", footer=s.footer, items=items,
                                  height=s.height_overview_cm, common=s.common_scale))

    # 3. Vues individuelles : une figure par zone et par indicateur
    for (zname, results), tag in zip(zones_valid, tags):
        for k, r in results.items():
            if r.ok:
                views.append(dict(group="single", caption=_join_title(k, zname if nz > 1 else ""),
                                  basename=f"{prefix}_" + (f"{tag}_" if nz > 1 else "") + safe_filename(k),
                                  title=s.title, subtitle=s.subtitle, zone_name=zname, footer=s.footer,
                                  items=[replace(r.indicator, heading=k, sub="")],
                                  height=s.height_single_cm, common=False))
    return views


def build_view(view, s):
    return build_figure(view["items"], s, view["height"], view["title"], view.get("subtitle", ""), view.get("zone_name", ""), view.get("footer", ""), view["common"])

## 5. Export et sauvegarde

In [ ]:
# ============================================================
# 5. EXPORT DES FIGURES ET SAUVEGARDE JSON
# ============================================================

def _bbox_kwargs(fig):
    """Recadrage 'tight' uniquement si un élément dépasse du cadre (sécurité anti-coupure)."""
    r = fig.canvas.get_renderer()
    tb = fig.get_tightbbox(r)
    w, h = fig.get_size_inches()
    tol = 0.01
    if tb.x0 < -tol or tb.y0 < -tol or tb.x1 > w + tol or tb.y1 > h + tol:
        return {"bbox_inches": "tight", "pad_inches": 0.08}
    return {}


def figure_bytes(fig, fmt="png", dpi=300, transparent=False):
    buf = io.BytesIO()
    with _rc():
        kw = _bbox_kwargs(fig)
        if transparent:
            fig.savefig(buf, format=fmt, dpi=dpi, transparent=True, **kw)
        else:
            fig.savefig(buf, format=fmt, dpi=dpi, facecolor="white", **kw)
    return buf.getvalue()


def safe_filename(text):
    t = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    t = re.sub(r"[^A-Za-z0-9_-]+", "_", t).strip("_")
    return t or "graphique"


def export_figure(fig, basename, s):
    """Écrit la figure dans tous les formats sélectionnés. Retourne la liste des chemins."""
    os.makedirs(s.export_dir, exist_ok=True)
    paths = []
    for fmt in s.formats:
        path = os.path.join(s.export_dir, f"{basename}.{fmt}")
        data = figure_bytes(fig, fmt=fmt, dpi=s.dpi, transparent=s.transparent)
        with open(path, "wb") as f:
            f.write(data)
        paths.append(os.path.abspath(path))
    return paths


def save_project_json(state, path):
    path = path.strip()
    if not path:
        raise ValueError("indiquez un nom de fichier JSON.")
    if not path.lower().endswith(".json"):
        path += ".json"
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    out = json.loads(json.dumps(state))   # copie profonde
    for zone in out["zones"]:
        for raw in zone["indicators"].values():
            v, err = parse_number(raw.get("target", ""))
            raw["target"] = v if err is None else raw.get("target", "")
            for c in raw["contributions"]:
                v, err = parse_number(c.get("value", ""))
                c["value"] = v if err is None else c.get("value", "")
    out["app_version"] = APP_VERSION
    out["saved_at"] = datetime.now().isoformat(timespec="seconds")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    return os.path.abspath(path)


def load_project_json(path):
    path = path.strip()
    if not path:
        raise ValueError("indiquez le chemin du fichier JSON.")
    if not os.path.isfile(path):
        raise FileNotFoundError(f"fichier introuvable : {os.path.abspath(path)}")
    with open(path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError as e:
            raise ValueError(f"JSON illisible (ligne {e.lineno}) : {e.msg}")
    return normalize_state(data), os.path.abspath(path)

## 5 bis. Échange avec Excel (copier-coller)

In [ ]:
# ============================================================
# 5 bis. ÉCHANGE AVEC EXCEL PAR COPIER-COLLER (modèle « clé | valeur »)
# ============================================================
# Collage : une clé par ligne (ex. « BbioChaud_widget »), puis une valeur par zone :
#   colonne B = 1re zone, colonne C = 2e zone, etc.
# Une ligne « zone_widget | Zone A | Zone B » (facultative) associe les colonnes aux zones par nom.
# Seules les clés collées sont mises à jour : on peut coller tout le tableau ou quelques lignes.

EXCEL_TITLE_KEY = "titre"
EXCEL_SUBTITLE_KEY = "soustitre"
EXCEL_FOOTER_KEY = "infos"
EXCEL_ZONE_KEY = "zone"
EXCEL_CEPNR_EQUALS_CEP_KEY = "cepnr_equals_cep"
# Clés du modèle reconnues mais sans équivalent dans ce notebook (signalées, ignorées)
EXCEL_IGNORED_KEYS = {"nbrange", "heightbar", "heightfond", "graduationx", "alphafond"}


def _excel_key(v):
    """Normalise une clé : minuscules, sans espaces ni suffixe _widget ; IcC_lot1 -> icc_lot01."""
    if not isinstance(v, str):
        return None
    k = v.strip().lower().replace(" ", "")
    if k.endswith("_widget"):
        k = k[:-7]
    m = re.fullmatch(r"icc_lot0*(\d+)", k)
    if m:
        k = f"icc_lot{int(m.group(1)):02d}"
    return k or None


def _known_keys():
    keys = {EXCEL_TITLE_KEY, EXCEL_SUBTITLE_KEY, EXCEL_FOOTER_KEY, EXCEL_ZONE_KEY, EXCEL_CEPNR_EQUALS_CEP_KEY} | EXCEL_IGNORED_KEYS | {f"dh_g{i}_nom" for i in range(1, 9)}
    for spec in EXCEL_MAP.values():
        keys |= {spec["show"], spec["max"]} | {c[0] for c in spec["contribs"]}
    return keys


def _excel_bool(v, default=True):
    s = str(v).strip().lower() if v is not None else ""
    if not s:
        return default
    if s in ("1", "oui", "o", "vrai", "true", "yes", "x"):
        return True
    if s in ("0", "non", "n", "faux", "false", "no"):
        return False
    num, err = parse_number(s)
    return (num != 0) if err is None else default


def parse_pasted_table(text):
    """Découpe un tableau collé depuis Excel. Retourne (lignes reconnues, lignes inconnues).
    Ligne reconnue : (clé normalisée, clé d'origine, [valeurs par colonne])."""
    known = _known_keys()
    rows, unknown = [], []
    for line in str(text or "").splitlines():
        if not line.strip():
            continue
        if "\t" in line:                      # copie Excel standard
            cells = line.split("\t")
        elif ";" in line:                     # CSV « français »
            cells = line.split(";")
        else:                                 # saisie manuelle « clé valeur »
            cells = line.strip().split(None, 1)
        cells = [c.strip().strip('"').strip() for c in cells]
        j = next((i for i, c in enumerate(cells) if c), None)
        if j is None:
            continue
        k = _excel_key(cells[j])
        vals = cells[j + 1:]
        while vals and vals[-1] == "":
            vals.pop()
        if k in known:
            rows.append((k, cells[j], vals))
        else:
            unknown.append(cells[j][:40])
    return rows, unknown


def _merge_contributions(key, raw, pairs, skip_zero, warnings, zname):
    """Met à jour les contributions d'un indicateur avec les clés collées.
    Les contributions non collées conservent leur valeur ; les libellés et couleurs
    déjà saisis sont conservés ; les contributions ajoutées à la main (sans clé) sont gardées à la fin."""
    spec = EXCEL_MAP[key]
    existing = raw["contributions"][:raw["n_active"]]
    by_key = {c["key"]: c for c in existing if c.get("key")}
    new = []
    for ck, label in spec["contribs"]:
        if ck in pairs:
            value = pairs[ck]
            v, err = parse_number(value)
            if err:
                warnings.append(f"{zname} / {key} — {excel_name(ck)} : {err} (à corriger dans l'onglet).")
        elif ck in by_key:
            value = by_key[ck]["value"]
            v, err = parse_number(value)
        else:
            continue
        if skip_zero and err is None and v == 0:
            continue
        base = by_key.get(ck, {})
        new.append({"key": ck, "label": base.get("label") or label, "value": str(value),
                    "color": base.get("color") or _spec_color(key, ck)})
    new += [c for c in existing if not c.get("key")]
    if not new:   # au moins une contribution (total nul)
        ck, label = spec["contribs"][0]
        new = [{"key": ck, "label": label, "value": "0", "color": _spec_color(key, ck)}]
    if len(new) > max_contrib(key):
        warnings.append(f"{zname} / {key} : plus de {max_contrib(key)} contributions, "
                        f"les dernières sont ignorées.")
    raw["contributions"] = _pad(new, key)
    raw["n_active"] = min(len(new), max_contrib(key))


def apply_pasted_table(state, text, start_zone=0, skip_zero=True):
    """Applique un tableau collé à l'état. start_zone : zone de la 1re colonne de valeurs
    (-1 = nouvelle zone). Retourne (nouvel état, compte rendu)."""
    rows, unknown = parse_pasted_table(text)
    if not rows:
        raise ValueError("aucune clé du modèle reconnue dans le texte collé "
                         "(ex. « Bbiomax_widget<TAB>180,7 »).")
    state = normalize_state(copy.deepcopy(state))
    zones = state["zones"]
    rep = {"updated": [], "created": [], "warnings": [], "ignored": [], "unknown": unknown}

    table = {}
    for k, orig, vals in rows:
        if k in table:
            rep["warnings"].append(f"Clé « {orig} » collée plusieurs fois : dernière ligne retenue.")
        table[k] = (orig, vals)
    rep["ignored"] = [table[k][0] for k in table if k in EXCEL_IGNORED_KEYS]

    if EXCEL_TITLE_KEY in table:
        t = next((v for v in table[EXCEL_TITLE_KEY][1] if v), "")
        if t:
            state["settings"]["title"] = t
            rep["updated"].append("Titre du graphique")
    if EXCEL_SUBTITLE_KEY in table:
        t = next((v for v in table[EXCEL_SUBTITLE_KEY][1] if v), "")
        state["settings"]["subtitle"] = t
        rep["updated"].append("Sous-titre")
    if EXCEL_FOOTER_KEY in table:
        t = next((v for v in table[EXCEL_FOOTER_KEY][1] if v), "")
        state["settings"]["footer"] = t
        rep["updated"].append("Informations de bas de graphique")

    data_keys = [k for k in table if k not in (EXCEL_TITLE_KEY, EXCEL_SUBTITLE_KEY, EXCEL_FOOTER_KEY, EXCEL_ZONE_KEY)
                 and k not in EXCEL_IGNORED_KEYS]
    ncols = max([len(table[k][1]) for k in data_keys] + [0])
    if ncols == 0 and not rep["updated"]:
        raise ValueError("clés reconnues mais aucune valeur : vérifiez que la colonne des "
                         "valeurs a bien été copiée.")

    names = table.get(EXCEL_ZONE_KEY, ("", []))[1]
    n_before = len(zones)
    touched_by_zone = {}
    base = len(zones) if start_zone is None or start_zone < 0 else min(start_zone, len(zones))
    for j in range(ncols):
        name = names[j].strip() if j < len(names) else ""
        if name:
            zi = next((i for i, z in enumerate(zones) if z["name"].casefold() == name.casefold()), None)
            if zi is None:
                zones.append(new_zone(name))
                zi = len(zones) - 1
                rep["created"].append(name)
        else:
            zi = base + j
            while zi >= len(zones):
                zones.append(new_zone(f"Zone {len(zones) + 1}"))
                rep["created"].append(zones[-1]["name"])
        zone = zones[zi]
        zname = zone["name"]
        pairs = {k: table[k][1][j] for k in data_keys if j < len(table[k][1]) and table[k][1][j] != ""}
        if not pairs:
            continue
        touched = []
        for key, spec in EXCEL_MAP.items():
            raw = zone["indicators"][key]
            changed = False
            if any(ck in pairs for ck, _ in spec["contribs"]):
                _merge_contributions(key, raw, pairs, skip_zero, rep["warnings"], zname)
                changed = True
            if spec["max"] in pairs:
                raw["target"] = pairs[spec["max"]]
                v, err = parse_number(raw["target"])
                if err or v <= 0:
                    rep["warnings"].append(f"{zname} / {key} — {excel_name(spec['max'])} : "
                                           f"valeur maximale invalide (à corriger dans l'onglet).")
                changed = True
            if spec["show"] in pairs:
                raw["include"] = _excel_bool(pairs[spec["show"]], True)
                changed = True
            if key == "DH":
                for gi in range(1, 9):
                    nk = f"dh_g{gi}_nom"
                    if nk in pairs and gi <= len(raw["contributions"]):
                        raw["contributions"][gi - 1]["label"] = pairs[nk].strip() or f"Groupe {gi}"
                        changed = True
            if changed:
                touched.append(key + ("" if raw["include"] else " (masqué)"))
        if _excel_bool(pairs.get(EXCEL_CEPNR_EQUALS_CEP_KEY), False):
            cep, cepnr = zone["indicators"]["CEP"], zone["indicators"]["CEPnr"]
            contribs = copy.deepcopy(cep["contributions"])
            for c in contribs:
                if c["key"].startswith("cep") and not c["key"].startswith("cepnr"):
                    c["key"] = "cepnr" + c["key"][3:]
            cepnr["contributions"], cepnr["n_active"] = contribs, cep["n_active"]
            rep["warnings"].append(f"{zname} : cepnr_equals_cep = 1, contributions du CEPnr recopiées "
                                   f"depuis le CEP (cible CEPnr inchangée).")
            if not any(t.startswith("CEPnr") for t in touched):
                touched.append("CEPnr")
        touched_by_zone.setdefault(zi, set()).update(t.split(" ")[0] for t in touched)
        if touched:
            rep["updated"].append(f"{zname} : " + ", ".join(touched))
    # Zones créées par le collage : seuls les indicateurs collés sont inclus
    for zi in range(n_before, len(zones)):
        for key in INDICATOR_KEYS:
            if key not in touched_by_zone.get(zi, set()):
                zones[zi]["indicators"][key]["include"] = False
    return state, rep


def _fmt_excel_value(v, sep):
    num, err = parse_number(v)
    if err:
        return str(v).strip()
    s = f"{num:.10g}"
    return s.replace(".", ",") if sep == "," else s


def state_to_excel_text(state, sep=","):
    """Tableau « clé | zone 1 | zone 2 … » des valeurs actuelles, à coller dans Excel.
    Retourne (texte, avertissements)."""
    zones = state["zones"]
    warnings = []
    lines = [["titre_widget", state["settings"].get("title", "")],
             ["sousTitre_widget", state["settings"].get("subtitle", "")],
             ["infos_widget", state["settings"].get("footer", "")],
             ["zone_widget"] + [z["name"] for z in zones]]
    for key, spec in EXCEL_MAP.items():
        per_zone = []
        for z in zones:
            raw = z["indicators"][key]
            active = raw["contributions"][:raw["n_active"]]
            per_zone.append({c["key"]: c["value"] for c in active if c.get("key")})
            manual = [c["label"] or "sans libellé" for c in active if not c.get("key")]
            if manual:
                warnings.append(f"{z['name']} / {key} : contribution(s) sans clé Excel non exportée(s) : "
                                + ", ".join(manual) + ".")
        for ck, _ in spec["contribs"]:
            lines.append([excel_name(ck)] + [_fmt_excel_value(pz.get(ck, "0"), sep) for pz in per_zone])
        if key == "DH":
            for gi in range(1, 9):
                vals = []
                for z in zones:
                    raw = z["indicators"]["DH"]
                    vals.append(raw["contributions"][gi-1].get("label", f"Groupe {gi}"))
                lines.append([f"DH_G{gi}_nom_widget"] + vals)
        lines.append([excel_name(spec["max"])]
                     + [_fmt_excel_value(z["indicators"][key]["target"], sep) for z in zones])
        lines.append([excel_name(spec["show"])]
                     + ["1" if z["indicators"][key]["include"] else "0" for z in zones])
    lines.append(["cepnr_equals_cep_widget"] + ["0"] * len(zones))
    return "\n".join("\t".join(r) for r in lines), warnings

# ---------- Fichier Excel XLSX : génération et lecture en mémoire

def excel_template_bytes(state=None):
    state = normalize_state(state or default_state())
    text, _ = state_to_excel_text(state, ",")
    rows = [line.split("\t") for line in text.splitlines()]
    wb = Workbook()
    ws = wb.active
    ws.title = "Saisie RE2020"
    for row in rows:
        ws.append(row)
    dark = "1F4E78"; light = "D9EAF7"; white = "FFFFFF"
    for cell in ws[1]:
        cell.font = Font(bold=True, color=white)
        cell.fill = PatternFill("solid", fgColor=dark)
    ws.freeze_panes = "B1"
    ws.column_dimensions["A"].width = 30
    for i in range(2, ws.max_column + 1):
        ws.column_dimensions[chr(64+i)].width = 22
    for row in range(1, ws.max_row + 1):
        ws.cell(row, 1).font = Font(bold=True)
        if row in (1,2,3,4):
            for col in range(1, ws.max_column + 1):
                ws.cell(row,col).fill = PatternFill("solid", fgColor=light)
        ws.cell(row,1).alignment = Alignment(wrap_text=True)
    info = wb.create_sheet("Mode d'emploi")
    info["A1"] = "Modèle d'import RE2020 Bullet Graphs v4.0"
    info["A1"].font = Font(bold=True, size=15, color=dark)
    instructions = [
        "Remplir uniquement les colonnes de zones dans l'onglet Saisie RE2020.",
        "titre_widget, sousTitre_widget et infos_widget sont des paramètres généraux.",
        "zone_widget définit le nom de chaque zone.",
        "Pour plusieurs groupes DH : DH_G1_widget à DH_G8_widget et les noms associés DH_G1_nom_widget à DH_G8_nom_widget.",
        "Les cellules vides ne remplacent pas les valeurs existantes lors d'un collage partiel.",
        "show_* = 1 affiche l'indicateur ; 0 le masque."
    ]
    for i,t in enumerate(instructions,3): info.cell(i,1).value=t
    info.column_dimensions["A"].width=110
    for row in info.iter_rows():
        for c in row: c.alignment=Alignment(wrap_text=True, vertical="top")
    buf=io.BytesIO(); wb.save(buf); return buf.getvalue()


def excel_upload_to_text(data):
    wb = load_workbook(io.BytesIO(data), data_only=True, read_only=True)
    ws = wb["Saisie RE2020"] if "Saisie RE2020" in wb.sheetnames else wb[wb.sheetnames[0]]
    lines=[]
    for row in ws.iter_rows(values_only=True):
        vals=["" if v is None else str(v) for v in row]
        while vals and vals[-1]=="": vals.pop()
        if vals: lines.append("\t".join(vals))
    return "\n".join(lines)


def download_link(data, filename, label):
    b64=base64.b64encode(data).decode("ascii")
    return f'<a download="{_html.escape(filename)}" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}" style="display:inline-block;padding:7px 12px;background:#1F4E78;color:white;text-decoration:none;border-radius:4px">{_html.escape(label)}</a>'


## 6. Interface utilisateur

In [ ]:
# ============================================================
# 6. INTERFACE UTILISATEUR (ipywidgets)
# ============================================================

_STYLE = {"description_width": "170px"}
_COLORS = {"error": "#B3261E", "warning": "#8A5A00", "success": "#1E6B3A", "info": "#1F4E79"}
_GROUPS = [("zone", "Vues d'ensemble (par zone)"),
           ("indicator", "Par indicateur (comparaison des zones)"),
           ("single", "Vues individuelles")]


def _msg_html(kind, title, items=()):
    color = _COLORS.get(kind, "#333")
    li = "".join(f"<li>{_html.escape(str(i))}</li>" for i in items)
    body = f"<ul style='margin:4px 0 0 18px;padding:0'>{li}</ul>" if li else ""
    return (f"<div style='border-left:4px solid {color};padding:6px 10px;margin:4px 0;"
            f"background:rgba(127,127,127,0.06)'><b style='color:{color}'>"
            f"{_html.escape(title)}</b>{body}</div>")


class IndicatorTab:
    """Onglet de saisie d'un indicateur (pour la zone affichée). Toutes les lignes existent
    toujours : masquer une ligne ne supprime pas sa saisie."""

    def __init__(self, key, on_change):
        self.key = key
        self.max = max_contrib(key)
        self._on_change = on_change
        self.include = W.Checkbox(value=True, description="Inclure dans les graphiques", indent=False)
        self.n = W.IntSlider(value=1, min=1, max=self.max, step=1,
                             description="Nombre de contributions", style=_STYLE,
                             layout=W.Layout(width="420px"))
        self.unit = W.Text(description="Unité", style=_STYLE, layout=W.Layout(width="420px"))
        self.target = W.Text(description="Valeur maximale (cible)", placeholder="ex. 63",
                             style=_STYLE, layout=W.Layout(width="420px"))
        self.keys, self.key_html = [""] * self.max, []
        self.labels, self.values, self.colors, self.rows = [], [], [], []
        header = W.HBox([
            W.HTML("<b>N°</b>", layout=W.Layout(width="34px")),
            W.HTML("<b>Clé Excel</b>", layout=W.Layout(width="140px")),
            W.HTML("<b>Libellé</b>", layout=W.Layout(width="340px")),
            W.HTML("<b>Valeur</b>", layout=W.Layout(width="110px")),
            W.HTML("<b>Couleur</b>", layout=W.Layout(width="180px")),
        ])
        for i in range(self.max):
            kh = W.HTML(layout=W.Layout(width="140px"))
            lab = W.Text(placeholder=f"Contribution {i + 1}", layout=W.Layout(width="340px"))
            val = W.Text(value="0", layout=W.Layout(width="110px"))
            col = W.ColorPicker(value=PALETTE[i % len(PALETTE)], concise=False, layout=W.Layout(width="180px"))
            row = W.HBox([W.HTML(f"{i + 1}", layout=W.Layout(width="34px")), kh, lab, val, col])
            self.key_html.append(kh); self.labels.append(lab); self.values.append(val)
            self.colors.append(col); self.rows.append(row)
        self.total_html = W.HTML()
        self.msg_html = W.HTML()
        self.widget = W.VBox([
            self.include, self.n, self.unit, self.target,
            W.HTML("<hr style='margin:6px 0'>"), header, *self.rows,
            W.HTML("<small>La clé Excel relie une contribution au tableau collé ; "
                   "les lignes sans clé sont saisies à la main.</small>"),
            W.HTML("<hr style='margin:6px 0'>"), self.total_html, self.msg_html,
        ], layout=W.Layout(padding="6px"))

        for w in [self.include, self.n, self.unit, self.target, *self.labels, *self.values, *self.colors]:
            w.observe(self._changed, names="value")
        self._update_rows()

    def _changed(self, change):
        if change["owner"] is self.n:
            self._update_rows()
        self._on_change(self)

    def _update_rows(self):
        for i, row in enumerate(self.rows):
            row.layout.display = "" if i < self.n.value else "none"

    def _show_key(self, i):
        k = self.keys[i]
        self.key_html[i].value = (f"<code style='font-size:11px'>{_html.escape(excel_name(k))}</code>"
                                  if k else "<span style='color:#999'>—</span>")

    def get_raw(self):
        return {
            "unit": self.unit.value,
            "target": self.target.value,
            "n_active": int(self.n.value),
            "include": bool(self.include.value),
            "contributions": [{"key": self.keys[i], "label": self.labels[i].value,
                               "value": self.values[i].value, "color": self.colors[i].value}
                              for i in range(self.max)],
        }

    def set_raw(self, raw):
        self.include.value = bool(raw.get("include", True))
        self.n.value = min(max(int(raw.get("n_active", 1)), 1), self.max)
        self.unit.value = str(raw.get("unit", ""))
        self.target.value = str(raw.get("target", ""))
        for i, c in enumerate(raw["contributions"][:self.max]):
            self.keys[i] = str(c.get("key", "") or "")
            self._show_key(i)
            self.labels[i].value = str(c.get("label", ""))
            self.values[i].value = str(c.get("value", ""))
            color, _ = normalize_color(c.get("color"), PALETTE[i % len(PALETTE)])
            self.colors[i].value = color
        self._update_rows()

    def refresh_feedback(self, s):
        """Met à jour l'aperçu du total et les messages de l'onglet. Retourne la validation."""
        res = validate_indicator(self.key, self.get_raw())
        if not self.include.value:
            self.total_html.value = "<i>Indicateur exclu des graphiques pour cette zone.</i>"
            self.msg_html.value = ""
            return res
        total, ok_vals = 0.0, True
        for i in range(self.n.value):
            v, err = parse_number(self.values[i].value)
            if err or v < 0:
                ok_vals = False
            else:
                total += v
        tgt, terr = parse_number(self.target.value)
        unit = _html.escape(self.unit.value.strip())
        if ok_vals:
            txt = f"<b>Total calculé : {fmt_num(total, s.decimals, s.decimal_sep)} {unit}</b>"
            if terr is None and tgt > 0:
                txt += (f" &nbsp;|&nbsp; Max : {fmt_target(tgt, s.decimals, s.decimal_sep)} {unit}"
                        f" &nbsp;|&nbsp; {fmt_num(100 * total / tgt, 0, s.decimal_sep)} % de la cible")
        else:
            txt = "<b>Total calculé : —</b> (valeurs à corriger)"
        self.total_html.value = txt
        if res.errors:
            self.msg_html.value = _msg_html("error", "À corriger", res.errors)
        elif res.warnings:
            self.msg_html.value = _msg_html("warning", "Remarques", res.warnings)
        else:
            self.msg_html.value = _msg_html("success", "Saisie valide")
        return res


class RE2020App:
    def __init__(self, state=None):
        self._loading = False
        self.zones = [new_zone("Zone 1")]
        self.cur = 0
        st = normalize_state(state or default_state())

        # --- Paramètres généraux
        self.w_title = W.Text(description="Titre", style=_STYLE, layout=W.Layout(width="700px"))
        self.w_subtitle = W.Text(description="Sous-titre", style=_STYLE, layout=W.Layout(width="700px"))
        self.w_footer = W.Textarea(description="Infos en bas", style=_STYLE, layout=W.Layout(width="700px", height="80px"))
        self.w_width = W.BoundedFloatText(min=8, max=60, step=0.5, description="Largeur (cm)",
                                          style=_STYLE, layout=W.Layout(width="300px"))
        self.w_h_over = W.BoundedFloatText(min=0, max=100, step=0.5, description="Hauteur ensemble (cm)",
                                           style=_STYLE, layout=W.Layout(width="300px"))
        self.w_h_single = W.BoundedFloatText(min=0, max=100, step=0.5, description="Hauteur individuelle (cm)",
                                             style=_STYLE, layout=W.Layout(width="300px"))
        self.w_font = W.IntSlider(min=6, max=20, description="Taille des textes (pt)", style=_STYLE,
                                  layout=W.Layout(width="420px"))
        self.w_show = W.Checkbox(description="Afficher les valeurs dans les segments", indent=False)
        self.w_hide0 = W.Checkbox(description="Masquer les contributions nulles dans la légende", indent=False)
        self.w_common = W.Checkbox(description="Échelle commune entre zones (vues par indicateur)", indent=False)
        self.w_dec = W.IntSlider(min=0, max=4, description="Nombre de décimales", style=_STYLE,
                                 layout=W.Layout(width="420px"))
        self.w_bg = W.ToggleButtons(options=[("Fond blanc", "blanc"), ("Fond transparent", "transparent")],
                                    description="Fond (export)", style=_STYLE)
        self.w_sep = W.ToggleButtons(options=[("Virgule", ","), ("Point", ".")],
                                     description="Séparateur décimal", style=_STYLE)
        general = W.VBox([
            self.w_title, self.w_subtitle, self.w_footer,
            W.HBox([self.w_width, self.w_h_over, self.w_h_single]),
            W.HTML("<small>Hauteur 0 = automatique (recommandé). Une hauteur trop faible est "
                   "automatiquement augmentée pour éviter les chevauchements.</small>"),
            self.w_font, self.w_dec, self.w_show, self.w_hide0, self.w_common, self.w_bg, self.w_sep,
        ])

        # --- Collage depuis Excel / copie vers Excel
        self.w_excel_template = W.HTML(value=download_link(excel_template_bytes(st), "Modele_import_RE2020_v4.xlsx", "Télécharger le modèle Excel"))
        self.w_excel_upload = W.FileUpload(accept=".xlsx", multiple=False, description="Importer un fichier Excel")
        self.btn_excel_import = W.Button(description="Lire le fichier Excel", icon="upload", button_style="success", layout=W.Layout(width="220px"))
        self.w_paste = W.Textarea(placeholder="Collez ici des lignes du tableau Excel (clé | valeur zone 1 | "
                                              "valeur zone 2 …)\nex. BbioChaud_widget\t48,8",
                                  layout=W.Layout(width="100%", height="180px"))
        self.w_paste_zone = W.Dropdown(description="1re colonne de valeurs →", style=_STYLE,
                                       layout=W.Layout(width="480px"))
        self.w_skip0 = W.Checkbox(value=True, indent=False,
                                  description="Retirer les contributions nulles des indicateurs mis à jour")
        self.btn_paste = W.Button(description="Remplir à partir du collage", icon="paste",
                                  button_style="success", layout=W.Layout(width="240px"))
        self.btn_paste_clear = W.Button(description="Vider", icon="eraser", layout=W.Layout(width="100px"))
        self.btn_to_excel = W.Button(description="Générer le tableau des valeurs actuelles", icon="table",
                                     layout=W.Layout(width="320px"))
        self.w_excel_out = W.Textarea(layout=W.Layout(width="100%", height="160px"),
                                      placeholder="Le tableau à copier dans Excel apparaîtra ici.")
        paste_box = W.VBox([
            W.HTML("<b>1. Modèle Excel</b><br><small>Téléchargez le modèle, remplissez-le puis importez-le ci-dessous.</small>"),
            self.w_excel_template, W.HBox([self.w_excel_upload, self.btn_excel_import]),
            W.HTML("<hr><b>2. Copier-coller Excel</b><br><small>Copiez dans Excel tout ou partie du tableau (colonne des clés + colonne(s) de "
                   "valeurs) puis collez-le ici. Seules les valeurs collées sont modifiées. "
                   "Colonne B = 1re zone, C = 2e zone… Une ligne <code>zone_widget</code> facultative "
                   "associe les colonnes aux zones par nom (zones créées si besoin).</small>"),
            self.w_paste, self.w_paste_zone, self.w_skip0,
            W.HBox([self.btn_paste, self.btn_paste_clear]),
            W.HTML("<hr style='margin:8px 0'><b>Sauvegarder les valeurs dans Excel</b><br><small>Générez le "
                   "tableau, cliquez dedans, Ctrl+A puis Ctrl+C, et collez-le en A1 dans Excel. "
                   "Libellés, couleurs et unités ne sont conservés que dans le JSON du projet.</small>"),
            self.btn_to_excel, self.w_excel_out,
        ])

        # --- Zones
        self.w_zone = W.Dropdown(description="Zone affichée", style={"description_width": "100px"},
                                 layout=W.Layout(width="380px"))
        self.w_zone_name = W.Text(description="Nom", style={"description_width": "40px"},
                                  layout=W.Layout(width="320px"))
        self.btn_zone_add = W.Button(description="Ajouter", icon="plus", layout=W.Layout(width="100px"))
        self.btn_zone_dup = W.Button(description="Dupliquer", icon="copy", layout=W.Layout(width="110px"))
        self.btn_zone_del = W.Button(description="Supprimer", icon="trash", button_style="warning",
                                     layout=W.Layout(width="110px"))
        zone_bar = W.HBox([self.w_zone, self.w_zone_name, self.btn_zone_add, self.btn_zone_dup,
                           self.btn_zone_del], layout=W.Layout(flex_flow="row wrap", align_items="center"))

        # --- Onglets indicateurs
        self.tabs = {k: IndicatorTab(k, self._on_tab_change) for k in INDICATOR_KEYS}
        self.tab_widget = W.Tab(children=[t.widget for t in self.tabs.values()])
        for i, k in enumerate(INDICATOR_KEYS):
            self.tab_widget.set_title(i, k)

        # --- Actions
        self.btn_generate = W.Button(description="Générer / actualiser les graphiques",
                                     button_style="primary", icon="refresh", layout=W.Layout(width="300px"))
        self.w_dir = W.Text(description="Dossier d'export", style=_STYLE, layout=W.Layout(width="420px"))
        self.w_prefix = W.Text(description="Préfixe des fichiers", style=_STYLE, layout=W.Layout(width="420px"))
        self.w_png = W.Checkbox(description="PNG", indent=False, layout=W.Layout(width="80px"))
        self.w_svg = W.Checkbox(description="SVG", indent=False, layout=W.Layout(width="80px"))
        self.w_pdf = W.Checkbox(description="PDF", indent=False, layout=W.Layout(width="80px"))
        self.w_dpi = W.BoundedIntText(min=300, max=1200, step=50, description="Résolution PNG (dpi)",
                                      style=_STYLE, layout=W.Layout(width="300px"))
        self.btn_exp_zone = W.Button(description="Exporter les vues d'ensemble", icon="download",
                                     layout=W.Layout(width="240px"))
        self.btn_exp_ind = W.Button(description="Exporter les vues par indicateur", icon="download",
                                    layout=W.Layout(width="260px"))
        self.btn_exp_single = W.Button(description="Exporter les vues individuelles", icon="download",
                                       layout=W.Layout(width="250px"))
        self.btn_exp_all = W.Button(description="Tout exporter", icon="download", button_style="info",
                                    layout=W.Layout(width="160px"))
        self.w_json = W.Text(value="projet_RE2020.json", description="Fichier projet (JSON)",
                             style=_STYLE, layout=W.Layout(width="560px"))
        self.btn_save = W.Button(description="Sauvegarder le projet", icon="save", layout=W.Layout(width="200px"))
        self.btn_load = W.Button(description="Charger le projet", icon="folder-open", layout=W.Layout(width="200px"))
        self.btn_demo = W.Button(description="Données de démonstration", icon="flask", layout=W.Layout(width="220px"))
        self.btn_new = W.Button(description="Projet vierge", icon="file", layout=W.Layout(width="160px"))

        export_box = W.VBox([
            W.HBox([self.w_dir, self.w_prefix]),
            W.HBox([W.HTML("<b>Formats :</b>&nbsp;", layout=W.Layout(width="80px")),
                    self.w_png, self.w_svg, self.w_pdf, self.w_dpi]),
            W.HBox([self.btn_exp_zone, self.btn_exp_ind, self.btn_exp_single, self.btn_exp_all],
                   layout=W.Layout(flex_flow="row wrap")),
        ])
        save_box = W.VBox([self.w_json, W.HBox([self.btn_save, self.btn_load, self.btn_demo, self.btn_new])])

        self.status = W.HTML()
        self.preview_boxes = {g: W.VBox() for g, _ in _GROUPS}
        self.preview_tab = W.Tab(children=list(self.preview_boxes.values()))
        for i, (_, title) in enumerate(_GROUPS):
            self.preview_tab.set_title(i, title)

        self.widget = W.VBox([
            W.HTML(f"<h3 style='margin:4px 0'>Graphiques RE2020 — bullet graphs "
                   f"<small style='color:#777'>v{APP_VERSION}</small></h3>"),
            W.Accordion(children=[paste_box, general],
                        titles=("Échange avec Excel (coller / copier)", "Paramètres généraux")),
            W.HTML("<h4 style='margin:10px 0 4px'>Zones RE2020 et saisie des indicateurs</h4>"),
            zone_bar,
            self.tab_widget,
            W.HTML("<h4 style='margin:10px 0 4px'>Actions</h4>"),
            self.btn_generate,
            W.Accordion(children=[export_box, save_box],
                        titles=("Export des graphiques", "Sauvegarde / chargement du projet")),
            self.status,
            W.HTML("<h4 style='margin:10px 0 4px'>Aperçu <small>(fond blanc, résolution réduite)</small></h4>"),
            self.preview_tab,
        ])

        actions = [(self.btn_generate, self.generate), (self.btn_exp_zone, self.export_zone),
                   (self.btn_exp_ind, self.export_indicator), (self.btn_exp_single, self.export_singles),
                   (self.btn_exp_all, self.export_all), (self.btn_save, self.save),
                   (self.btn_load, self.load), (self.btn_demo, self.load_demo),
                   (self.btn_new, self.new_project), (self.btn_paste, self.paste_import),
                   (self.btn_paste_clear, self.paste_clear), (self.btn_to_excel, self.to_excel),
                   (self.btn_excel_import, self.import_excel_file),
                   (self.btn_zone_add, self.zone_add), (self.btn_zone_dup, self.zone_duplicate),
                   (self.btn_zone_del, self.zone_delete)]
        self._buttons = [b for b, _ in actions]
        for b, f in actions:
            b.on_click(self._safe(f))
        for w in [self.w_dec, self.w_sep]:
            w.observe(lambda ch: self._refresh_all_feedback(), names="value")
        self.w_zone.observe(self._on_zone_select, names="value")
        self.w_zone_name.observe(self._on_zone_rename, names="value")

        self.set_state(st)

    # ---------- utilitaires
    def _safe(self, func):
        @functools.wraps(func)
        def wrapper(*_):
            for b in self._buttons:
                b.disabled = True
            try:
                func()
            except Exception as e:   # message lisible plutôt qu'une trace d'erreur
                self._status("error", "Erreur inattendue", [f"{type(e).__name__} : {e}"])
            finally:
                for b in self._buttons:
                    b.disabled = False
                self.btn_zone_del.disabled = len(self.zones) < 2
        return wrapper

    def _status(self, kind, title, items=()):
        self.status.value = _msg_html(kind, title, items)

    def _settings_raw(self):
        return {
            "title": self.w_title.value, "subtitle": self.w_subtitle.value, "footer": self.w_footer.value, "width_cm": self.w_width.value,
            "height_overview_cm": self.w_h_over.value, "height_single_cm": self.w_h_single.value,
            "font_size": self.w_font.value, "show_values": self.w_show.value,
            "hide_zero_legend": self.w_hide0.value, "common_scale": self.w_common.value,
            "decimals": self.w_dec.value, "background": self.w_bg.value,
            "decimal_sep": self.w_sep.value, "dpi": self.w_dpi.value,
            "export_dir": self.w_dir.value, "file_prefix": self.w_prefix.value,
            "formats": [f for f, w in (("png", self.w_png), ("svg", self.w_svg), ("pdf", self.w_pdf)) if w.value],
        }

    # ---------- zones
    def _sync_current(self):
        """Recopie la saisie des onglets dans la zone affichée."""
        z = self.zones[self.cur]
        z["name"] = self.w_zone_name.value.strip() or f"Zone {self.cur + 1}"
        z["indicators"] = {k: t.get_raw() for k, t in self.tabs.items()}

    def _zone_has_errors(self, z):
        return any(raw["include"] and not validate_indicator(k, raw).ok
                   for k, raw in z["indicators"].items())

    def _refresh_zone_options(self, follow=False):
        """Met à jour les listes de zones. follow : la cible du collage suit la zone affichée."""
        self._loading = True
        try:
            opts = [(f"{i + 1}. {z['name']}" + (" ⚠" if self._zone_has_errors(z) else ""), i)
                    for i, z in enumerate(self.zones)]
            self.w_zone.options = opts
            self.w_zone.value = self.cur
            prev = self.w_paste_zone.value
            self.w_paste_zone.options = ([(f"{i + 1}. {z['name']}", i) for i, z in enumerate(self.zones)]
                                         + [("Nouvelle zone", -1)])
            valid = [o[1] for o in self.w_paste_zone.options]
            self.w_paste_zone.value = self.cur if (follow or prev not in valid) else prev
        finally:
            self._loading = False
        self.btn_zone_del.disabled = len(self.zones) < 2

    def _load_zone(self, i):
        self.cur = min(max(i, 0), len(self.zones) - 1)
        z = self.zones[self.cur]
        self._loading = True
        try:
            self.w_zone_name.value = z["name"]
            for k, t in self.tabs.items():
                t.set_raw(z["indicators"][k])
        finally:
            self._loading = False
        self._refresh_all_feedback()

    def _on_zone_select(self, change):
        if self._loading or change["new"] is None:
            return
        self._sync_current()
        self._load_zone(change["new"])
        self._refresh_zone_options(follow=True)

    def _on_zone_rename(self, change):
        if self._loading:
            return
        self.zones[self.cur]["name"] = change["new"].strip() or f"Zone {self.cur + 1}"
        self._refresh_zone_options()

    def zone_add(self):
        self._sync_current()
        self.zones.append(new_zone(f"Zone {len(self.zones) + 1}"))
        self._load_zone(len(self.zones) - 1)
        self._refresh_zone_options(follow=True)
        self._status("info", f"Zone « {self.zones[self.cur]['name']} » ajoutée : saisissez ou collez ses valeurs.")

    def zone_duplicate(self):
        self._sync_current()
        z = copy.deepcopy(self.zones[self.cur])
        z["name"] += " (copie)"
        self.zones.insert(self.cur + 1, z)
        self._load_zone(self.cur + 1)
        self._refresh_zone_options(follow=True)
        self._status("info", f"Zone dupliquée : « {z['name']} ».")

    def zone_delete(self):
        if len(self.zones) < 2:
            self._status("error", "Impossible de supprimer la dernière zone.")
            return
        name = self.zones.pop(self.cur)["name"]
        self._load_zone(min(self.cur, len(self.zones) - 1))
        self._refresh_zone_options(follow=True)
        self._status("info", f"Zone « {name} » supprimée.")

    # ---------- état
    def get_state(self):
        self._sync_current()
        return {"version": 2, "settings": self._settings_raw(),
                "zones": copy.deepcopy(self.zones), "current_zone": self.cur}

    def set_state(self, state):
        st = normalize_state(state)
        s, _ = validate_settings(st["settings"])
        self._loading = True
        try:
            self.w_title.value = s.title
            self.w_subtitle.value = s.subtitle
            self.w_footer.value = s.footer
            self.w_width.value = s.width_cm
            self.w_h_over.value = s.height_overview_cm
            self.w_h_single.value = s.height_single_cm
            self.w_font.value = s.font_size
            self.w_show.value = s.show_values
            self.w_hide0.value = s.hide_zero_legend
            self.w_common.value = s.common_scale
            self.w_dec.value = s.decimals
            self.w_bg.value = s.background
            self.w_sep.value = s.decimal_sep
            self.w_dpi.value = s.dpi
            self.w_dir.value = s.export_dir
            self.w_prefix.value = s.file_prefix
            self.w_png.value, self.w_svg.value, self.w_pdf.value = (f in s.formats for f in ("png", "svg", "pdf"))
        finally:
            self._loading = False
        self.zones = st["zones"]
        self._load_zone(st.get("current_zone", 0))
        self._refresh_zone_options(follow=True)

    def _on_tab_change(self, tab):
        if self._loading:
            return
        s, _ = validate_settings(self._settings_raw())
        self._update_tab_title(tab, tab.refresh_feedback(s))

    def _update_tab_title(self, tab, res):
        i = INDICATOR_KEYS.index(tab.key)
        flag = "" if (res.ok or not tab.include.value) else " ⚠"
        excl = " (exclu)" if not tab.include.value else ""
        self.tab_widget.set_title(i, f"{tab.key}{flag}{excl}")

    def _refresh_all_feedback(self):
        if self._loading:
            return
        s, _ = validate_settings(self._settings_raw())
        for t in self.tabs.values():
            self._update_tab_title(t, t.refresh_feedback(s))

    def _collect(self):
        """Valide toutes les zones. Retourne (settings, [(zone, {indicateur: résultat})], erreurs, avertissements)."""
        self._sync_current()
        s, warns = validate_settings(self._settings_raw())
        self._refresh_all_feedback()
        self._refresh_zone_options()
        multi = len(self.zones) > 1
        zones_valid, errors = [], []
        for z in self.zones:
            results = {}
            for k in INDICATOR_KEYS:
                raw = z["indicators"][k]
                if raw["include"]:
                    results[k] = validate_indicator(k, raw)
                    for e in results[k].errors:
                        errors.append((f"{z['name']} / " if multi else "") + f"{k} — {e}")
            zones_valid.append((z["name"], results))
        if not any(zr for _, zr in zones_valid):
            warns.append("Aucun indicateur inclus.")
        return s, zones_valid, errors, warns

    # ---------- graphiques
    def _image(self, fig):
        data = figure_bytes(fig, fmt="png", dpi=PREVIEW_DPI, transparent=False)
        w_px = int(fig.get_size_inches()[0] * PREVIEW_DPI)
        return W.Image(value=data, format="png",
                       layout=W.Layout(width=f"{w_px}px", max_width="100%", height="auto",
                                       border="1px solid #e0e0e0"))

    def _empty_message(self, group, zones_valid, errors):
        if group == "indicator" and len(zones_valid) < 2:
            return _msg_html("info", "Ajoutez au moins une deuxième zone pour comparer un indicateur entre zones.")
        if errors:
            return _msg_html("error", "Aucune vue générée : saisies à corriger (voir le message d'état).")
        return _msg_html("info", "Aucune vue à afficher.")

    def generate(self):
        s, zones_valid, errors, notes = self._collect()
        views = plan_views(zones_valid, s)
        noted = set()
        for g, _ in _GROUPS:
            children = []
            for v in (v for v in views if v["group"] == g):
                fig, note = build_view(v, s)
                if note and g not in noted:
                    notes.append(f"{dict(_GROUPS)[g]} : {note}.")
                    noted.add(g)
                children += [W.HTML(f"<b>{_html.escape(v['caption'])}</b>"), self._image(fig)]
            self.preview_boxes[g].children = children or [W.HTML(self._empty_message(g, zones_valid, errors))]
        blocked = [n for n, r in zones_valid if r and any(not x.ok for x in r.values())]
        if blocked:
            notes.append("Vue d'ensemble non générée pour : " + ", ".join(blocked) + ".")
        stamp = f"{datetime.now():%H:%M:%S}"
        if errors:
            self._status("error", f"Graphiques actualisés à {stamp}, des saisies sont à corriger.", errors + notes)
        elif notes:
            self._status("warning", f"Graphiques actualisés à {stamp}.", notes)
        else:
            self._status("success", f"Graphiques actualisés à {stamp} ({len(views)} figure(s)).")
        return not errors

    def _export(self, groups):
        s, zones_valid, errors, notes = self._collect()
        if not s.formats:
            self._status("error", "Aucun format d'export sélectionné (PNG, SVG ou PDF).")
            return
        views = [v for v in plan_views(zones_valid, s) if v["group"] in groups]
        if "indicator" in groups and len(zones_valid) < 2:
            notes.append("Vues par indicateur : au moins deux zones sont nécessaires.")
        paths = []
        for v in views:
            fig, note = build_view(v, s)
            if note:
                notes.append(f"{v['caption']} : {note}.")
            paths += export_figure(fig, v["basename"], s)
        items = [f"Fichier : {p}" for p in paths] + errors + notes
        if not paths:
            self._status("error", "Rien n'a été exporté.", items)
        elif errors:
            self._status("warning", f"Export partiel : {len(paths)} fichier(s) écrit(s), "
                                    f"les éléments à corriger ont été ignorés.", items)
        else:
            self._status("success", f"Export terminé : {len(paths)} fichier(s) écrit(s) "
                                    f"(fond {s.background}, PNG {s.dpi} dpi).", items)

    def export_zone(self):
        self._export({"zone"})

    def export_indicator(self):
        self._export({"indicator"})

    def export_singles(self):
        self._export({"single"})

    def export_all(self):
        self._export({"zone", "indicator", "single"})

    # ---------- Excel
    def import_excel_file(self):
        if not self.w_excel_upload.value:
            self._status("error", "Aucun fichier Excel sélectionné.")
            return
        uploaded = next(iter(self.w_excel_upload.value)) if isinstance(self.w_excel_upload.value, (tuple, list)) else next(iter(self.w_excel_upload.value.values()))
        data = uploaded["content"] if isinstance(uploaded, dict) else uploaded.content
        self.w_paste.value = excel_upload_to_text(bytes(data))
        self.paste_import()

    def paste_import(self):
        if not self.w_paste.value.strip():
            self._status("error", "La zone de collage est vide.")
            return
        try:
            state, rep = apply_pasted_table(self.get_state(), self.w_paste.value,
                                            self.w_paste_zone.value, self.w_skip0.value)
        except ValueError as e:
            self._status("error", "Collage non interprété.", [str(e)])
            return
        cur_before = self.cur
        state["current_zone"] = cur_before
        self.set_state(state)
        ok = self.generate()
        items = [f"Mis à jour — {x}" for x in rep["updated"]]
        items += [f"Zone créée — {x}" for x in rep["created"]]
        items += rep["warnings"]
        if rep["ignored"]:
            items.append("Clés non gérées par ce notebook (ignorées) : " + ", ".join(rep["ignored"]))
        if rep["unknown"]:
            items.append("Lignes non reconnues (ignorées) : " + ", ".join(rep["unknown"][:12])
                         + (" …" if len(rep["unknown"]) > 12 else ""))
        if not ok:
            items.append("Certaines saisies sont à corriger : voir les zones et onglets marqués ⚠.")
        kind = "success" if ok and not rep["warnings"] and not rep["unknown"] else "warning"
        self._status(kind, "Valeurs collées appliquées, graphiques actualisés.", items)

    def paste_clear(self):
        self.w_paste.value = ""

    def to_excel(self):
        st = self.get_state()
        text, warns = state_to_excel_text(st, self.w_sep.value)
        self.w_excel_out.value = text
        self._status("warning" if warns else "success",
                     "Tableau généré : copiez-le (Ctrl+A, Ctrl+C) et collez-le en A1 dans Excel.", warns)

    # ---------- projet
    def save(self):
        path = save_project_json(self.get_state(), self.w_json.value)
        self._status("success", "Projet sauvegardé.", [path])

    def load(self):
        try:
            state, path = load_project_json(self.w_json.value)
        except (OSError, ValueError) as e:
            self._status("error", "Chargement impossible.", [str(e)])
            return
        self.set_state(state)
        self.generate()
        self._status("success", "Projet chargé et graphiques actualisés.", [path])

    def load_demo(self):
        self.set_state(demo_state())
        self.generate()
        self._status("info", "Données de démonstration (fictives, deux zones) chargées.")

    def new_project(self):
        self.set_state(default_state())
        for box in self.preview_boxes.values():
            box.children = []
        self._status("info", "Projet vierge : collez ou saisissez les valeurs puis cliquez sur "
                             "« Générer / actualiser ».")

    def show(self):
        display(self.widget)

## 7. Lancement de l'interface

In [ ]:
# ============================================================
# 7. LANCEMENT DE L'INTERFACE
# ============================================================
app = RE2020App(default_state())
app.generate()
app.show()